# **\[Video Games System Recommendation] - Ananta Boemi Adji**

### **Latar Belakang**

Industri video game merupakan salah satu sektor hiburan terbesar dan terus berkembang pesat dari tahun ke tahun. Jumlah game yang tersedia di pasaran sangat banyak, baik dari berbagai genre, platform, maupun rating yang berbeda-beda. Kondisi ini membuat para pemain (gamer) sering kali kebingungan untuk menentukan game mana yang layak untuk dimainkan, terutama yang sesuai dengan preferensi mereka.

Permasalahan utama yang ingin diselesaikan adalah **bagaimana membantu pengguna menemukan video game yang sesuai dengan preferensi mereka secara efisien dan personal**. Oleh karena itu, pada proyek ini akan dikembangkan sebuah **sistem rekomendasi video game** berdasarkan data penjualan, rating pengguna, rating kritikus, serta metadata lainnya seperti genre dan platform.

Sistem rekomendasi akan dibangun menggunakan dua pendekatan utama, yaitu **Content-Based Filtering** dan **Collaborative Filtering**. Hasil akhir dari sistem ini akan menyajikan rekomendasi Top-N game kepada pengguna berdasarkan kesamaan konten dan pola perilaku pengguna lain.

---

## **Tujuan Proyek**

Tujuan utama dari proyek ini adalah:

* Mengembangkan sistem rekomendasi game yang dapat memberikan rekomendasi personal kepada pengguna.
* Mengeksplorasi dua pendekatan rekomendasi: Content-Based Filtering dan Collaborative Filtering.
* Melatih dan mengevaluasi model untuk menghasilkan rekomendasi Top-N game.
* Memberikan insight dari fitur-fitur yang paling mempengaruhi minat pengguna terhadap game tertentu.
* Menyediakan laporan analisis dan sistem yang mudah dipahami dan dapat digunakan oleh pengguna akhir.

---

## **Informasi Dataset**

Dataset yang digunakan dalam proyek ini adalah **Video Game Sales with Ratings**, yang tersedia secara publik di platform Kaggle.

* **Nama Dataset:** Video Game Sales with Ratings
* **Sumber:** Kaggle
* **Publisher:** rush4ratio
* **Link Dataset:** [https://www.kaggle.com/datasets/rush4ratio/video-game-sales-with-ratings](https://www.kaggle.com/datasets/rush4ratio/video-game-sales-with-ratings)

Dataset ini menggabungkan data penjualan video game dari VGChartz dan data rating dari Metacritic, mencakup berbagai platform dan genre.

---

## **Alur Pengerjaan Project**

Langkah-langkah pengerjaan proyek ini meliputi:

1. **Exploratory Data Analysis (EDA):**

   * Mengeksplorasi distribusi penjualan, rating, genre, dan platform.
   * Visualisasi korelasi antar fitur dan deteksi outlier/missing values.
   * Menentukan fitur penting yang dapat digunakan untuk rekomendasi.

2. **Preprocessing:**

   * Penanganan nilai hilang dan duplikat.
   * Encoding fitur kategorikal dan normalisasi numerik.
   * Pembuatan matriks kemiripan untuk content-based filtering dan user-item matrix untuk collaborative filtering.

3. **Modeling:**

   * Membangun sistem rekomendasi dengan **Content-Based Filtering** (berdasarkan kemiripan genre, platform, rating).
   * Membangun sistem rekomendasi dengan **Collaborative Filtering** menggunakan pendekatan matrix factorization (misalnya SVD).
   * Menyajikan hasil **Top-N Recommendation** untuk setiap pendekatan.

4. **Interpretasi dan Insight:**

   * Menganalisis hasil rekomendasi dan kecocokan dengan data historis.
   * Membandingkan kelebihan dan kekurangan dari kedua pendekatan.

---

## **Penilaian yang Diharapkan**

Proyek ini diharapkan dapat menghasilkan beberapa poin penting:

* Menyediakan sistem rekomendasi yang dapat memberikan **Top-N game terbaik** untuk pengguna berdasarkan preferensi historis dan konten game.
* Mampu mengevaluasi performa model dengan metrik seperti **Precision\@K**, **Recall\@K**, atau **RMSE** (untuk collaborative filtering).
* Menunjukkan pemahaman mendalam terhadap proses data preparation dan pemilihan fitur.
* Memberikan laporan analitis dan sistematis yang menjelaskan alur pembuatan sistem rekomendasi.
* Menyediakan dua pendekatan model dan evaluasi komparatif terhadap hasilnya.

# **1. Import Library**

Pada bagian ini saya akan berfokus import semua library yang akan dibutuhkan pada project ini.

| Kategori Library            | Library / Fungsi                              | Kegunaan                                                                  |
| ------------------- | --------------------------------------------- | ------------------------------------------------------------------------- |
| Data Manipulation   | `pandas`, `numpy`                             | Membaca, membersihkan, dan memproses data tabular                         |
| Visualisasi         | `matplotlib.pyplot`, `seaborn`                | Visualisasi distribusi data, korelasi, dan insight EDA                    |
| Preprocessing       | `MinMaxScaler`, `cosine_similarity`           | Normalisasi data numerik dan perhitungan kemiripan untuk content-based    |
| NLP (Content-Based) | `TfidfVectorizer`                             | Mengubah data teks menjadi vektor numerik (misalnya genre game)           |
| Collaborative       | `surprise`, `SVD`, `Reader`, `cross_validate` | Algoritma rekomendasi berbasis user-rating dengan collaborative filtering |
| Evaluation          | `mean_squared_error`, `sqrt`                  | Menghitung metrik RMSE untuk evaluasi rekomendasi collaborative           |
| Warnings Handling   | `warnings`                                    | Menyembunyikan warning yang tidak penting                                 |

---

Pada project ini juga saya akan menggunakan lightfm karena compatible dengan google colab.

In [ ]:
!pip install lightfm --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gdown
import ipywidgets as widgets

from IPython.display import display, clear_output
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

from lightfm import LightFM
from lightfm.data import Dataset as LFDataset
from lightfm.evaluation import precision_at_k, recall_at_k

import warnings
warnings.filterwarnings('ignore')

# import re
# from nltk.stem import WordNetLemmatizer
# from nltk.corpus import stopwords
# import nltk

# **2. Load Dataset**

Dataset dimuat dari Google Drive menggunakan link share ID, dataset saya download langsung dari kaggle.

In [ ]:
url = 'https://drive.google.com/uc?id=1jAOTErV-OG6mwBPzAywgpHdlsiWRcUjl'
df = pd.read_csv(url)

# **3. Exploratory Data Analysis (EDA)**

### **Melihat ringkasan dataset**

Pertama saya akan melihat ringkasan dari dataset seperti:
- statistik untuk kolom numerik: rata-rata, standar deviasi, min, max, dll.
- Serta melihat struktur umum dataset: jumlah baris & kolom, tipe data tiap kolom, dan nilai non-null

In [ ]:
df.head()

df.info()
df.describe()

**Insight**:

1. **Jumlah Data & Missing Value**

   * Dataset memiliki **16.719 entri**.
   * Ada beberapa kolom dengan **missing value cukup besar**, terutama:

     * `Critic_Score` (\~51% missing)
     * `Critic_Count` (\~51% missing)
     * `User_Score` (\~40% missing, dan bertipe object)
     * `User_Count` (\~54% missing)
     * `Developer` (\~40% missing)
     * `Rating` (\~41% missing)
       → data terkait ulasan **tidak selalu tersedia** untuk semua game.

3. **Distribusi Penjualan**

   * Sebagian besar game memiliki **penjualan yang sangat kecil**
   * Artinya, hanya sedikit game yang menjadi **top seller**

4. **Global Sales**

   * Median **Global\_Sales** adalah **0.17 juta unit**

5. **Review Metrics**

   * Rata-rata **Critic\_Score** adalah **68.97** → sebagian besar game **di-review cukup baik**.
   * Namun, data `User_Score` harus diolah karena masih berupa object.

6. **Lainnya**
  * Dataset ini memiliki **distribusi penjualan yang sangat tidak merata** → mayoritas game tidak terlalu laku, hanya sedikit yang blockbuster.
  * Banyak **missing value** di data review → perlu dilakukan pengaturan pada saat preprocessing.
  * Penjualan game dan review score berpotensi punya **hubungan menarik untuk dianalisis** (misal: apakah review score berkorelasi dengan sales?).

---

  **Struktur Dataset**

Struktur dan informasi yang ada pada dataset:

| Nama Kolom        | Deskripsi                                          | Tipe Data | Rentang / Contoh             |
| ----------------- | -------------------------------------------------- | --------- | ---------------------------- |
| `Name`            | Nama video game                                    | String    | "Super Mario Bros."          |
| `Platform`        | Platform rilis game (PS2, X360, PC, dll.)          | String    | "PS4", "X360", "PC"          |
| `Year_of_Release` | Tahun rilis game                                   | Integer   | 1980 - 2020 (banyak missing) |
| `Genre`           | Genre game                                         | String    | "Action", "Shooter"          |
| `Publisher`       | Perusahaan penerbit game                           | String    | "Nintendo", "EA Sports"      |
| `NA_Sales`        | Penjualan di Amerika Utara (dalam juta unit)       | Float     | 0.00 - 41.49                 |
| `EU_Sales`        | Penjualan di Eropa (dalam juta unit)               | Float     | 0.00 - 29.02                 |
| `JP_Sales`        | Penjualan di Jepang (dalam juta unit)              | Float     | 0.00 - 10.22                 |
| `Other_Sales`     | Penjualan di wilayah lain                          | Float     | 0.00 - 10.57                 |
| `Global_Sales`    | Total penjualan global                             | Float     | 0.01 - 82.74                 |
| `Critic_Score`    | Skor dari kritikus (0 - 100)                       | Float     | 13.0 - 98.0                  |
| `Critic_Count`    | Jumlah kritikus yang menilai game                  | Integer   | 3 - 51                       |
| `User_Score`      | Skor dari pengguna Metacritic (dalam skala 0 - 10) | Float     | 0.0 - 9.8                    |
| `User_Count`      | Jumlah pengguna yang memberikan skor               | Integer   | 1 - 10665                    |
| `Developer`       | Pengembang game                                    | String    | "Ubisoft", "EA Canada"       |
| `Rating`          | Rating ESRB game (misalnya E, T, M)                | String    | "E", "T", "M"                |

### **Mengecek missing values dan duplikasi dari dataset**

In [ ]:
df.isnull().sum()
df.duplicated().sum()

**Insight**:

Sudah aman tidak ada indikasi adanya data missing dan duplikat

### **Visualisasi Distribusi Data Numerik**

In [ ]:
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
sns.histplot(df['User_Score'].dropna(), kde=True, bins=20)
plt.title('Distribusi User Score')

plt.subplot(1, 3, 2)
sns.histplot(df['Critic_Score'].dropna(), kde=True, bins=20)
plt.title('Distribusi Critic Score')

plt.subplot(1, 3, 3)
sns.histplot(df['Global_Sales'].dropna(), kde=True, bins=20)
plt.title('Distribusi Global Sales')

plt.tight_layout()
plt.show()

**Insight**:

Berikut **deskripsi singkat** untuk setiap bagian:

---

Distribusi **User Score**

* **Banyak noise/data kotor** → terlihat ada nilai `tbd` atau label non-numerik.
* Distribusi belum dapat dianalisis dengan baik sebelum dilakukan pembersihan.
* Indikasi bahwa data User Score masih perlu preprocessing.

---

2️Distribusi **Critic Score**

* Distribusi mendekati **normal** dengan sedikit skew ke kiri.
* Mayoritas game mendapatkan skor **60–85**.
* Jarang ada game dengan skor sangat rendah (<40) atau sangat tinggi (>90).
* Menunjukkan bahwa **kritikus cenderung memberikan penilaian di rentang menengah ke atas**.

---

Distribusi **Global Sales**

* Distribusi sangat **right-skewed** (long-tail effect).
* Mayoritas game memiliki penjualan **di bawah 1 juta unit**.
* Hanya sedikit game yang mencapai penjualan **fenomenal** (>10 juta unit).

### **Check Outlier dengan Boxplot**

In [ ]:
plt.figure(figsize=(15, 4))

plt.subplot(1, 3, 1)
sns.boxplot(x=df['User_Score'])
plt.title('Boxplot User Score')

plt.subplot(1, 3, 2)
sns.boxplot(x=df['Critic_Score'])
plt.title('Boxplot Critic Score')

plt.subplot(1, 3, 3)
sns.boxplot(x=df['Global_Sales'])
plt.title('Boxplot Global Sales')

plt.tight_layout()
plt.show()

**Insight**:

- Data User Score menunjukkan banyak outlier dan kemungkinan data kotor (perlu preprocessing lebih lanjut). Critic Score terdistribusi normal dengan sebagian besar skor berada di rentang 60–80, namun tetap terdapat beberapa outlier di skor rendah.
- Global Sales, distribusi sangat right-skewed dengan banyak game berpenjualan rendah dan hanya sedikit game yang mencapai penjualan sangat tinggi (outlier), menunjukkan pola umum di industri hiburan di mana segelintir game blockbuster mendominasi pasar.

### **Korelasi antara fitur numerik**

In [ ]:
df['User_Score'] = pd.to_numeric(df['User_Score'], errors='coerce')

numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns

plt.figure(figsize=(8, 6))
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='coolwarm')
plt.title('Korelasi antar fitur numerik')
plt.show()

**Insight**:

Terlihat bahwa Global Sales sangat berkorelasi kuat dengan semua regional sales, terutama dengan NA_Sales (0.94), diikuti oleh EU_Sales (0.90) dan Other_Sales (0.75), yang memang wajar karena Global Sales merupakan akumulasi dari penjualan regional. Korelasi antara Critic Score dan User Score bersifat positif (0.58), mengindikasikan adanya hubungan cukup baik antara penilaian kritikus dan penilaian pengguna. Fitur Year_of_Release memiliki korelasi lemah terhadap hampir semua variabel, tahun rilis game tidak secara langsung memengaruhi sales atau skor. Penjualan lebih dipengaruhi oleh performa di regional market dibanding faktor-faktor seperti skor.

### **Cek Unik Value pada Fitur Kategorikal**

In [ ]:
categorical_cols = ['Platform', 'Genre', 'Developer', 'Publisher', 'Rating']
for col in categorical_cols:
    print(f"{col} - Jumlah kategori unik: {df[col].nunique()}")
    print(df[col].value_counts().head(), "\n")

**Insight**:

Industri game ini didominasi oleh beberapa platform populer (PS2, DS), genre action dan sports, serta beberapa developer dan publisher besar seperti Ubisoft dan EA. Pasar game ini cenderung fokus pada game dengan rating yang lebih ramah untuk berbagai usia, namun juga memiliki segmen game dewasa yang cukup besar.

### **Visualisasi Genre / Platform Populer**

In [ ]:
plt.figure(figsize=(12,5))
sns.countplot(data=df, y='Genre', order=df['Genre'].value_counts().index[:10])
plt.title("10 Genre Terpopuler")
plt.show()

plt.figure(figsize=(12,5))
sns.countplot(data=df, y='Platform', order=df['Platform'].value_counts().index[:10])
plt.title("10 Platform Terpopuler")
plt.show()

**Insight**:

-  Genre Action adalah genre game paling populer dengan jumlah rilis terbanyak, diikuti oleh Sports dan Misc. Ini menunjukkan bahwa game dengan elemen aksi dan olahraga memiliki daya tarik yang cukup besar di pasar.

- Untuk platform, PS2 dan DS mendominasi sebagai platform dengan jumlah game terbanyak, menandakan kedua konsol tersebut dalam industri game. Konsol seperti PS3, Wii, dan X360 juga cukup tinggi, memperlihatkan dominasi konsol generasi tersebut dalam mendistribusikan game.

### **Distribusi Game Berdasarkan Tahun Rilis**

In [ ]:
plt.figure(figsize=(12,5))
sns.histplot(df['Year_of_Release'].dropna(), bins=30, kde=False)
plt.title("Distribusi Tahun Rilis Game")
plt.xlabel("Tahun")
plt.ylabel("Jumlah Game")
plt.show()

**Insight**:

Jumlah perilisan game mengalami peningkatan signifikan mulai awal 2000-an dan mencapai puncaknya sekitar tahun 2008–2009, yang merupakan era tertinggu untuk banyak konsol populer seperti PS2, Wii, dan Xbox 360. Setelah itu, terjadi penurunan bertahap dalam jumlah game yang dirilis, yang kemungkinan disebabkan oleh pergeseran industri ke arah kualitas, digital distribution, atau perubahan tren pengembangan game. Tahun-tahun sebelum 1995 menunjukkan jumlah rilis yang masih sangat rendah, menandakan awal pertumbuhan industri game.

# **4. Preprocessing / Data Preparation**

### **Menangani missing values dan persiapan dataset**

Sebelum masuk ke tahap lainnya beberapa hal yang perlu diperhatikan seperti:
- Drop baris yang masih memiliki missing value di kolom penting untuk digunakan dalam projek rekomendasi sistem ini.
- Menghapus kolom yang tidak terlalu kritikal untuk modeling jika terlalu banyak missing
- Ubah kolom 'User_Score' menjadi numerik, tbd → NaN

In [ ]:
df = df.drop(['Critic_Score', 'Critic_Count', 'User_Count', 'Developer'], axis=1)

df['User_Score'] = pd.to_numeric(df['User_Score'], errors='coerce')
df.dropna(subset=['Year_of_Release', 'Genre', 'Publisher', 'User_Score', 'Rating'], inplace=True)
df['Year_of_Release'] = df['Year_of_Release'].astype(int)

### **Konversi tipe data**

Kolom `Year_of_Release` perlu untuk di ubah ke bentuk `int`

In [ ]:
df['Year_of_Release'] = df['Year_of_Release'].astype(int)

### **Standarisasi format dan penyatuan data**

Karena disini ada pendekatan content-based filtering, maka perlu menyatukan metadata ke dalam 1 kolom gabungan, selanjutnya akan dilakukan TF-IDF vectorization.

In [ ]:
df['metadata'] = (
    df['Name'].fillna('') + ' ' +
    (df['Genre'].fillna('') + ' ') * 5 +
    df['Platform'].fillna('') + ' ' +
    df['Publisher'].fillna('') + ' ' +
    df['Rating'].fillna('') + ' ' +
    df['Year_of_Release'].fillna('').astype(str)
)

### **Normalisasi Fitur Numerik**

`User_Score` dan `Global_Sales` dinormalisasi untuk digunakan sebagai bobot tambahan.

In [ ]:
scaler = MinMaxScaler()
df[['User_Score_norm', 'Global_Sales_norm']] = scaler.fit_transform(df[['User_Score', 'Global_Sales']])

In [ ]:
df['User_Score_category'] = pd.cut(
    df['User_Score'], bins=[0, 5, 7, 10], labels=['low', 'medium', 'high']
).astype(str)

### **Simulasi Data User (untuk Collaborative Filtering)**

Dataset yang saya gunakan tidak memiliki user_id, maka dari itu saya menggunakan simulasi dengan asumsi game yang memiliki rating tinggi disukai oleh kebanyakan user (dummy).

In [ ]:
# --- Tambahkan ini di bagian preprocessing, sebelum membuat df_cb dan df_cf ---
df_filtered_for_cbf = df[df['User_Score'] >= 7].copy()
# Atau, jika Anda hanya ingin game dengan penjualan tinggi juga
# df_filtered_for_cbf = df[(df['User_Score'] >= 7) & (df['Global_Sales_norm'] > 0.5)].copy()

# Kemudian, gunakan df_filtered_for_cbf sebagai sumber untuk membuat df_cb:
df_cb = df_filtered_for_cbf[['Name', 'Genre', 'Platform', 'Publisher', 'Rating', 'Year_of_Release',
            'User_Score', 'Global_Sales']].copy()
# Dan lanjutkan preprocessing df_cb seperti biasa

In [ ]:
df_cf = df.copy()
df_cf = df_cf[df_cf['User_Score'] >= 7]

np.random.seed(42)
df_cf['user_id'] = np.random.randint(0, 500, size=len(df_cf))

df_cf = df_cf[['user_id', 'Name', 'User_Score']]

### **Check Feature yang digunakan**

- cb = Content-Based Filtering, penambahan feature gabungan untuk bisa meningkatkan hasil precision dan juga recall.

Serta perlu dilakukan reset df_cb index agar bisa sinkron dengan cosine sim.

- cf = Collaborative Filtering

In [ ]:
# # --- Bagian Baru: Fungsi Pembersihan Teks ---
# try:
#     nltk.data.find('corpora/wordnet')
# except nltk.downloader.DownloadError:
#     nltk.download('wordnet')
# try:
#     nltk.data.find('corpora/stopwords')
# except nltk.downloader.DownloadError:
#     nltk.download('stopwords')

# lemmatizer = WordNetLemmatizer()
# stop_words_nltk = set(stopwords.words('english'))

# def clean_text(text):
#     text = str(text).lower()
#     text = re.sub(r'[^a-z0-9\s]', '', text) # Hapus karakter non-alphanumeric
#     words = text.split()
#     words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words_nltk]
#     return ' '.join(words)

# # Terapkan pembersihan pada kolom-kolom yang akan digabungkan
# df_cb['Name_cleaned'] = df_cb['Name'].apply(clean_text)
# df_cb['Genre_cleaned'] = df_cb['Genre'].apply(clean_text)
# df_cb['Platform_cleaned'] = df_cb['Platform'].apply(clean_text)
# df_cb['Publisher_cleaned'] = df_cb['Publisher'].apply(clean_text)
# df_cb['Rating_cleaned'] = df_cb['Rating'].apply(clean_text)

# # --- Akhir Bagian Baru ---

In [ ]:
df_cb = df[['Name', 'Genre', 'Platform', 'Publisher', 'Rating', 'Year_of_Release',
            'User_Score', 'User_Score_norm', 'Global_Sales_norm']].copy()

df_cb['User_Score_category'] = pd.cut(df_cb['User_Score'], bins=[0, 5, 7, 10], labels=['low', 'medium', 'high']).astype(str)
df_cb['Global_Sales_category'] = pd.qcut(df_cb['Global_Sales_norm'], 3, labels=['low', 'medium', 'high'])
# Bagian yang perlu diganti: df_cb['combined_features']
# Code Pengganti (menggunakan kolom _cleaned dan bobot yang lebih moderat):
# Modifikasi df_cb['combined_features'] dengan bobot yang lebih agresif
# Perhatikan peningkatan faktor pengganda
df_cb['combined_features'] = (
    df_cb['Name'].fillna('') + ' ' +
    (df_cb['Genre'].fillna('') + ' ') * 8 +  # Bobot Genre sangat ditingkatkan
    (df_cb['Platform'].fillna('') + ' ') * 6 + # Bobot Platform sangat ditingkatkan
    df_cb['Publisher'].fillna('') + ' ' +
    df_cb['Rating'].fillna('') + ' ' +
    (df_cb['User_Score_category'].astype(str) + ' ') * 6 + # Bobot User Score Category sangat ditingkatkan
    (df_cb['Global_Sales_category'].astype(str) + ' ') * 6 + # Bobot Global Sales Category sangat ditingkatkan
    df_cb['Year_of_Release'].astype(str)
)

df_cb.reset_index(drop=True, inplace=True)

In [ ]:
df_cf = df_cf[['user_id', 'Name', 'User_Score']]

# **5. Modeling**

Dalam project ini saya akan lebih berfokus untuk pengembangan model dalam pendekatan content-based filtering karena penggunaan dataset yang lebih sesuai, bila menggunakan collaborative filtering masih kurang sesuai karena kurangnya feature ID user dalam dataset tersebut.

## **Pendekatan 1: Content-Based Filtering**

Pendekatan ini akan memberikan rekomendasi berdasarkan kemiripan metadata antar game kepada pengguna nantinya.

Tahap yang dilakukan:
- Sinkronkan semuanya antara df_cb dan juga df_cf
- Memperbarui fungsi
- Melakukan modeling

**Membuat ulang TF-IDF dan cosine similarity**

Kemudian melakukan mapping game ke indeks

In [ ]:
# Optimasi TfidfVectorizer dengan parameter yang lebih agresif
tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=30000,  # Tingkatkan max_features lebih lanjut
    ngram_range=(1, 4),  # Gunakan n-gram hingga 4 kata
    min_df=2,            # Tetapkan min_df yang rendah
    max_df=0.7           # Sesuaikan max_df untuk filtering kata yang sangat umum
)
tfidf_matrix = tfidf.fit_transform(df_cb['combined_features'])

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
indices = pd.Series(df_cb.index, index=df_cb['Name']).drop_duplicates()

# --- Tambahkan ini di bagian modeling, setelah cosine_sim untuk TF-IDF dihitung ---

# Hitung similarity berbasis numerik
# Pastikan kolom numerik yang digunakan adalah yang sudah dinormalisasi
numeric_features = df_cb[['User_Score_norm', 'Global_Sales_norm']]
# Isi NaN dengan 0 atau rata-rata jika ada, untuk menghindari kesalahan pada perhitungan similarity
numeric_features = numeric_features.fillna(0) # Mengisi NaN jika ada, pastikan tidak ada setelah preprocessing

# Gunakan cosine_similarity untuk fitur numerik juga
# Matriks harus 2D.
similarity_numeric = cosine_similarity(numeric_features)

# Gabungkan similarity teks dan numerik
# Eksperimen dengan bobot (0.7, 0.3) atau (0.8, 0.2) atau lainnya
# Sum of weights should be 1.0 for simplicity, or adjust as needed.
alpha = 0.75 # Bobot untuk similarity teks
beta = 0.25 # Bobot untuk similarity numerik
# --- Integrasi Langsung Fitur Numerik ke Perhitungan Similarity ---
# Hitung similarity berbasis numerik
numeric_features = df_cb[['User_Score_norm', 'Global_Sales_norm']].fillna(0)
similarity_numeric = cosine_similarity(numeric_features)

# Gabungkan similarity teks dan numerik
# Eksperimen dengan bobot alpha dan beta.
# Coba lebih banyak bobot pada tekstual (alpha > beta) atau sebaliknya
alpha = 0.85 # Bobot untuk similarity teks (coba lebih tinggi)
beta = 0.15  # Bobot untuk similarity numerik
cosine_sim_combined = (cosine_similarity(tfidf_matrix) * alpha) + (similarity_numeric * beta)

# Gunakan 'cosine_sim_combined' untuk evaluasi dan rekomendasi CBF
indices = pd.Series(df_cb.index, index=df_cb['Name']).drop_duplicates()
# Setelah baris ini, gunakan 'cosine_sim_combined' sebagai ganti 'cosine_sim'
# di fungsi run_cbf_recommendation dan evaluate_cbf.
# Contoh:
# cbf_precision, cbf_recall = evaluate_cbf(df_cb, df_cf, cosine_sim_combined, indices, k=5)
# print(f'Precision@5: {cbf_precision:.4f}')
# print(f'Recall@5: {cbf_recall:.4f}')

# --- Akhir penambahan ---

**Membuat fungsi rekomendasi game untuk pendekatan CBF**

In [ ]:
def run_cbf_recommendation(df_cb, cosine_sim, indices, title_input=None, k=5, verbose=True):
    if title_input is None:
        title_input = input("Masukkan judul game yang Anda sukai (CBF): ").strip()

    if title_input not in indices:
        if verbose:
            print("⚠️ Game tidak ditemukan dalam dataset Content-Based Filtering.")
        return pd.DataFrame()

    idx = indices[title_input]
    sim_scores = list(enumerate(cosine_sim[idx].ravel()))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:k+1]

    game_indices = [i[0] for i in sim_scores if i[0] < len(df_cb)]

    recommendations_df = df_cb.iloc[game_indices].copy()

    if 'User_Score' in recommendations_df.columns:
        recommendations_df = recommendations_df[['Name', 'User_Score']]
    else:
        recommendations_df = recommendations_df[['Name']]

    recommendations_df = recommendations_df.drop_duplicates(subset='Name').reset_index(drop=True)

    if verbose:
        print(f"\n📌 Rekomendasi game mirip dengan '{title_input}' (CBF):\n")
        display(recommendations_df)

    return recommendations_df

Fungsi evaluasi rekomendasi game untuk pendekatan CBF, untuk melihat bagaimana performa model menggunakan parameter precision dan recall.

In [ ]:
def evaluate_cbf(df_cb, df_cf, cosine_sim, indices, k=5):
    precision_list = []
    recall_list = []

    evaluated_titles = set(df_cb['Name'].unique()) & set(df_cf['Name'].unique())

    for title in list(evaluated_titles):
        title = str(title)  # pastikan string

        if title not in indices.index:
            continue

        idx = indices[title]
        idx = indices[title]
        idx = idx if np.isscalar(idx) else idx.iloc[0]

        sim_row = cosine_sim[idx]

        if hasattr(sim_row, 'toarray'):
            sim_row = sim_row.toarray().ravel()
        else:
            sim_row = np.array(sim_row).ravel()

        sim_scores = list(enumerate(sim_row))
        sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
        recommended_indices = [i for i, _ in sim_scores if i != idx][:k]

        try:
            recommended_titles = df_cb.iloc[recommended_indices]['Name'].dropna().tolist()
        except:
            continue

        relevant_users = df_cf[df_cf['Name'] == title]['user_id'].dropna().unique().tolist()
        relevant_items = df_cf[df_cf['user_id'].isin(relevant_users)]['Name'].dropna().unique().tolist()

        if len(recommended_titles) == 0 or len(relevant_items) == 0:
            continue

        true_positives = len(set(recommended_titles) & set(relevant_items))
        precision = true_positives / k
        recall = true_positives / len(relevant_items)

        precision_list.append(precision)
        recall_list.append(recall)

    if len(precision_list) == 0:
        return 0.0, 0.0

    return np.mean(precision_list), np.mean(recall_list)


**Contoh penggunaan Fungsi**

Disini pengguna menginputkan judul sebuah game yang mereka sukai, fungsi akan mencarikan beberapa game lainnya yang memiliki similiarity dari game tersebut.

In [ ]:
cbf_precision, cbf_recall = evaluate_cbf(df_cb, df_cf, cosine_sim, indices, k=5)
print(f'Precision@5: {cbf_precision:.4f}')
print(f'Recall@5: {cbf_recall:.4f}')


**Insight**:

- Precision@5: 0.0573
- Recall@5: 0.0168

Dari setiap 5 rekomendasi yang diberikan oleh sistem Content-Based Filtering, hanya sekitar 0.35% yang relevan bagi pengguna. Sistem hanya mampu menangkap sekitar 0.18% dari seluruh item yang seharusnya direkomendasikan kepada pengguna berdasarkan preferensi historis mereka.

Hasil ini mengindikasikan bahwa pendekatan berbasis konten (fitur Genre, Platform, Publisher) belum cukup kuat untuk menangkap selera pengguna secara akurat jika dibandingkan dengan pendekatan lain seperti Collaborative Filtering.

## **Pendekatan 2: Collaborative Filtering (LightFM)**

Tahap yang dilakukan:
- Buat dataset user-item untuk LightFM
- Latih model dengan loss 'warp' (Weighted Approximate-Rank Pairwise)
- Evaluasi menggunakan Precision@k dan Recall@k

In [ ]:
lfm_dataset = LFDataset()
lfm_dataset.fit(df_cf['user_id'], df_cf['Name'])

(interactions, weights) = lfm_dataset.build_interactions(
    [(row['user_id'], row['Name'], row['User_Score']) for _, row in df_cf.iterrows()]
)

model = LightFM(loss='warp')
model.fit(interactions, epochs=10, num_threads=2)

train_precision_cf = precision_at_k(model, interactions, k=5).mean()
train_recall_cf = recall_at_k(model, interactions, k=5).mean()

**Testing Evaluation**

Melihat hasil dari precision dan juga recall dari pendekatan Collaborative Filtering dengan LightFM.

In [ ]:
print(f'Precision@5: {train_precision_cf:.4f}')
print(f'Recall@5: {train_recall_cf:.4f}')

**Insight**:
- CF Precision@5: 0.2320
- CF Recall@5: 0.1196

Dari setiap 5 rekomendasi yang dihasilkan oleh sistem Collaborative Filtering, memiliki tingkat ke relevansi sekitar 22.52% bagi pengguna berdasarkan riwayat interaksi mereka.

Selain itu, sistem ini berhasil menangkap sekitar 11.84% dari seluruh item yang seharusnya direkomendasikan, model cukup mampu memahami pola preferensi antar pengguna.

Dibandingkan dengan Content-Based Filtering, pendekatan ini memberikan hasil yang lebih baik dan menunjukkan potensi dalam memberikan rekomendasi yang personal dan tepat sasaran, khususnya ketika data interaksi pengguna tersedia dalam jumlah cukup.

**Fungsi rekomendasi game untuk pendekatan CF**

Bagian ini yang akan digunakan untuk melakukan sistem rekomendasi dan juga akan digunakan sebagai penilaian akhir dari performa model.

In [ ]:
def run_cf_recommendation(model, dataset, df, k=10, min_score=5.0):
    game_title = input("Masukkan judul game yang Anda sukai (CF): ").strip()

    if game_title not in df['Name'].values:
        print("⚠️ Game tidak ditemukan dalam dataset Collaborative Filtering.")
        return pd.DataFrame()

    liked_users = df[(df['Name'] == game_title) & (df['User_Score'] >= min_score)]['user_id'].unique()

    if len(liked_users) == 0:
        print("⚠️ Tidak ada user yang menyukai game ini (dengan skor tinggi).")
        return pd.DataFrame()

    selected_user = np.random.choice(liked_users)
    n_items = len(dataset.mapping()[2])

    scores = model.predict(user_ids=np.repeat(selected_user, n_items), item_ids=np.arange(n_items))
    top_items = np.argsort(-scores)[:k]

    item_map = {v: k for k, v in dataset.mapping()[2].items()}
    game_names = [item_map[i] for i in top_items]

    recommendations_df = (
        df[df['Name'].isin(game_names)]
        .drop_duplicates(subset='Name')
        [['Name', 'User_Score']]
        .reset_index(drop=True)
    )

    print(f"\n📌 Rekomendasi berdasarkan preferensi user untuk '{game_title}' (CF):\n")
    display(recommendations_df)

    return recommendations_df

## **Game Recommendation System - Game Library**

Untuk bisa mengecek game apa saja yang terdapat pada recommentadion system ini bisa dengan menggunakan fungsi pencarian dibawah ini. Game yang terdapat pada fungsi Game Library ini nantinya bisa digunakan untuk mengecek fungsi recommend_games_cf dan recommend_games_cbf.

---

**Note**: Hal ini disebabkan karena dataset yang saya gunakan memang tidak mempunyai semua data game yang ada pada saat ini, terutama game-game yang baru rilis beberapa tahun belakangan.

In [ ]:
def get_available_games_by_genre_paginated(df_cb, df_cf, page_size=10):
    available_genres = sorted(df_cb['Genre'].dropna().unique())

    genre_dropdown = widgets.Dropdown(
        options=available_genres,
        description='🎮 Genre:',
        layout=widgets.Layout(width='50%')
    )

    output = widgets.Output()
    next_button = widgets.Button(description='Next ▶️')
    prev_button = widgets.Button(description='◀️ Previous')
    page_label = widgets.Label()

    state = {
        'filtered_df': pd.DataFrame(),
        'current_page': 1,
        'total_pages': 1
    }

    def update_output():
        with output:
            clear_output(wait=True)
            df = state['filtered_df']
            current_page = state['current_page']
            total_pages = state['total_pages']

            start_idx = (current_page - 1) * page_size
            end_idx = start_idx + page_size
            display(df.iloc[start_idx:end_idx].reset_index(drop=True))
            page_label.value = f"📄 Halaman {current_page} dari total {total_pages}"

    def on_genre_change(change):
        genre_input = change['new']
        common_games = set(df_cb['Name']).intersection(set(df_cf['Name']))
        filtered_df = df_cb[
            (df_cb['Genre'].str.contains(genre_input, case=False, na=False)) &
            (df_cb['Name'].isin(common_games))
        ][['Name', 'Genre', 'Publisher', 'Rating']].drop_duplicates().sort_values('Name').reset_index(drop=True)

        if filtered_df.empty:
            with output:
                clear_output()
                print(f"\n⚠️ Tidak ada game ditemukan untuk genre: {genre_input} ⚠️")
            state['filtered_df'] = pd.DataFrame()
            state['current_page'] = 1
            state['total_pages'] = 1
            page_label.value = ""
            return

        state['filtered_df'] = filtered_df
        state['current_page'] = 1
        state['total_pages'] = (len(filtered_df) + page_size - 1) // page_size
        update_output()

    def on_next_clicked(b):
        if state['current_page'] < state['total_pages']:
            state['current_page'] += 1
            update_output()

    def on_prev_clicked(b):
        if state['current_page'] > 1:
            state['current_page'] -= 1
            update_output()

    genre_dropdown.observe(on_genre_change, names='value')
    next_button.on_click(on_next_clicked)
    prev_button.on_click(on_prev_clicked)

    navigation = widgets.HBox([prev_button, next_button, page_label])
    display(genre_dropdown, navigation, output)

**Games Dataset**

karena dataset yang digunakan tidak memiliki seluruh data game yang ada saat ini, jadi disini bisa melihat list apa saja game yang terdapat dataset ini. Misalnya bisa ambil satu game yang terdapat pada dataset ini untuk digunakan dalam fungsi rekomendasi baik untuk Content-Based Filtering dan juga Collaborative Filtering.

In [ ]:
get_available_games_by_genre_paginated(df_cb, df_cf)

**System Recommendation Test**

Masukkan judul game untuk mendapatkan rekomendasi game serupa baik dengan Content-Based Filtering (CBF) dan juga Collaborative Filtering (CF)

In [ ]:
# run_cbf_recommendation(df_cb, cosine_sim, indices)

In [ ]:
# run_cf_recommendation(model, lfm_dataset, df_cf, k=5)

**Catatan**:

Bisa terjadi sebuah kondisi judul game yang dimasukan tidak ditemukan karena memang tidak ada pada dataest yang digunakan atau juga karena hanya game yang memiliki interaksi (user_id sintetis) dan metadata lengkap yang masuk ke dalam pemodelan.

# **6. Evaluation**

# Evaluasi Model Rekomendasi Game

## A. Metrik Evaluasi yang Digunakan

Dalam proyek ini, untuk mengukur kinerja sistem rekomendasi, digunakan dua metrik utama:

### Precision\@k

Precision\@k mengukur proporsi rekomendasi yang relevan di antara k item teratas yang direkomendasikan.
**Formula:**

$$
\text{Precision@k} = \frac{\text{Jumlah item relevan dalam top k rekomendasi}}{k}
$$

Precision tinggi berarti sistem dapat memberikan rekomendasi yang sesuai dengan preferensi pengguna.

### Recall\@k

Recall\@k mengukur proporsi item relevan yang berhasil direkomendasikan dari semua item relevan yang ada.
**Formula:**

$$
\text{Recall@k} = \frac{\text{Jumlah item relevan dalam top k rekomendasi}}{\text{Total item relevan yang tersedia}}
$$

Recall tinggi berarti sistem mampu menemukan sebagian besar item relevan untuk pengguna.

---

## B. Hasil Evaluasi

| Model                   | Precision\@5 | Recall\@5 |
| ----------------------- | ------------ | --------- |
| Content-Based Filtering | 0.0035       | 0.0018    |
| Collaborative Filtering | 0.2320       | 0.1196    |

### Interpretasi Hasil:

* **Content-Based Filtering** menghasilkan nilai **Precision\@5 sebesar 0.35%** dan **Recall\@5 sebesar 0.18%**, yang menunjukkan bahwa pendekatan ini **belum mampu memberikan rekomendasi yang relevan secara konsisten** berdasarkan metadata game saja. Hal ini bisa terjadi karena keterbatasan informasi yang digunakan (seperti genre, platform, dan publisher).

* **Collaborative Filtering (menggunakan LightFM dengan loss WARP)** menunjukkan performa yang **jauh lebih baik**, dengan **Precision\@5 sebesar 23.20%** dan **Recall\@5 sebesar 11.96%**. Ini menunjukkan bahwa sistem dapat merekomendasikan game yang **lebih relevan dan sesuai dengan preferensi pengguna**, karena pendekatan ini memanfaatkan pola interaksi antar pengguna (user-item interactions) yang lebih personal.

## C. Visualisasi Evaluasi

Berikut contoh visualisasi perbandingan metrik Precision\@5 dan Recall\@5 kedua model:

In [ ]:
models = ['Content-Based Filtering', 'Collaborative Filtering']
precision = [0.0035, 0.2320]
recall = [0.0018, 0.1196]

x = range(len(models))

plt.figure(figsize=(8,5))
plt.bar(x, precision, width=0.4, label='Precision@5', align='center', color='skyblue')
plt.bar(x, recall, width=0.4, label='Recall@5', align='edge', color='lightgreen')
plt.xticks(x, models)
plt.ylabel('Score')
plt.title('Comparison of Precision@5 and Recall@5 for Recommendation Models')
plt.legend()
plt.ylim(0, 0.3)
plt.show()

**Collaborative Filtering secara signifikan lebih baik dalam metrik evaluasi dibanding Content-Based Filtering.**

# **7. Project Summary and Results**

**Ringkasan Tahapan Proyek:**

1. **Eksplorasi Data (EDA)**
   Memahami struktur dataset video game, mengidentifikasi missing value, distribusi data, dan kolom penting.

2. **Preprocessing Data**

   * Menangani missing values dengan drop kolom dan baris yang kurang lengkap.
   * Konversi tipe data dan filtering berdasarkan User\_Score.
   * Membuat dataset baru dengan user\_id sintetis untuk Collaborative Filtering.

3. **Modeling**

   * Content-Based Filtering menggunakan TF-IDF dan cosine similarity dari metadata game.
   * Collaborative Filtering menggunakan model LightFM dengan loss WARP dan interaksi user-item berbobot.

4. **Evaluasi Model**
   Menghitung Precision\@5 dan Recall\@5 untuk kedua pendekatan.

---

**Kesimpulan Akhir Evaluasi:**

* **Collaborative Filtering lebih unggul** dalam memberikan rekomendasi game yang relevan berdasarkan interaksi pengguna, meskipun user dibuat secara sintetis.

* **Content-Based Filtering kurang efektif** dalam konteks dataset ini karena keterbatasan fitur metadata dan kompleksitas preferensi pengguna yang tidak tertangkap baik oleh metode ini.

---

**Potensi Perbaikan dan Pengembangan Selanjutnya:**

1. **Peningkatan Data User**
   Gunakan data interaksi pengguna asli (rating, review, klik) untuk Collaborative Filtering agar hasil lebih realistis.

2. **Pengayaan Fitur Content-Based**
   Tambahkan fitur lain seperti deskripsi game, tags, atau ulasan pengguna untuk memperkaya representasi metadata.

3. **Hybrid Recommender System**
   Gabungkan Content-Based dan Collaborative Filtering untuk memanfaatkan kelebihan kedua metode sekaligus.

4. **Hyperparameter Tuning dan Model Lain**
   Eksperimen dengan model lain seperti Matrix Factorization atau Neural Collaborative Filtering.

---

**Referensi**

* Dataset:
  [Video Game Sales Dataset](https://www.kaggle.com/datasets/rush4ratio/video-game-sales-with-ratings) oleh Rush Kirubi.

* Library & Tools:

  * Scikit-learn (TF-IDF, cosine similarity)
  * LightFM ([https://making.lyst.com/lightfm/docs/home.html](https://making.lyst.com/lightfm/docs/home.html)) untuk Collaborative Filtering
  * Pandas, Numpy untuk preprocessing dan analisis data

* Literatur dan Panduan:
  * Online tutorials dan dokumentasi LightFM